In [1]:
import celltypist
import py8rds
import anndata as ad


# 1. Load your reference file in backed mode (uses almost 0 RAM)
ref_adata = ad.read_h5ad("/media/sofia/TOSHIBA EXT/developing_human_brain/BTS_atlas.h5ad", backed="r")

# 2. Load your query dataset
query_adata = py8rds.seurat2adata(
  "/home/sofia/Projects/etmr/defaria/snRNA/data/processed/preprocessed/tumor_normalized.qs2",
  layer = "data"
  )
# Path to your query AnnData file and your downloaded model pkl file
model_pkl_path = "/media/sofia/TOSHIBA EXT/developing_human_brain/BTS_atlas_CellTypist_model.pkl"

# Run cell type annotation
predictions = celltypist.annotate(
  query_adata, 
  model=model_pkl_path, 
  majority_voting=True  # Optional: refines predictions using over-clustering
)

# Extract the annotated AnnData object with predictions added to obs metadata
adata_annotated = predictions.adata



/home/sofia/miniconda3/envs/sc_env/lib/python3.10/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv
Reading /home/sofia/Projects/etmr/defaria/snRNA/data/processed/preprocessed/tumor_normalized.qs2
Done reading /home/sofia/Projects/etmr/defaria/snRNA/data/processed/preprocessed/tumor_normalized.qs2
Concatenating split layers: ['data.GSM8058157', 'data.GSM8058158', 'data.GSM8058159', 'data.GSM8058160', 'data.GSM8058161', 'data.GSM8058162', 'data.GSM8058163', 'data.GSM8058164', 'data.GSM8058165']
🔬 Input data has 45423 cells and 18082 genes
🔗 Matching reference genes in the model
🧬 892 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
/home/sofia/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please 

In [3]:
# 1. Look at the CellTypist prediction table directly
print(predictions.predicted_labels.head())

                                                 predicted_labels  \
GSM8058157_KK22-H-225_AAACAAGCATTCGGTTACTACTCA-1               C4   
GSM8058157_KK22-H-225_AAACCAGGTTGCATCGACTACTCA-1               C4   
GSM8058157_KK22-H-225_AAACGGGCAATAACCGACTACTCA-1               C3   
GSM8058157_KK22-H-225_AAACTGGGTGCACAAAACTACTCA-1              C36   
GSM8058157_KK22-H-225_AAAGATGCAATCGTGAACTACTCA-1               C6   

                                                 over_clustering  \
GSM8058157_KK22-H-225_AAACAAGCATTCGGTTACTACTCA-1               0   
GSM8058157_KK22-H-225_AAACCAGGTTGCATCGACTACTCA-1               1   
GSM8058157_KK22-H-225_AAACGGGCAATAACCGACTACTCA-1               5   
GSM8058157_KK22-H-225_AAACTGGGTGCACAAAACTACTCA-1               6   
GSM8058157_KK22-H-225_AAAGATGCAATCGTGAACTACTCA-1               8   

                                                 majority_voting  
GSM8058157_KK22-H-225_AAACAAGCATTCGGTTACTACTCA-1              C4  
GSM8058157_KK22-H-225_AAACCAGGTTGCATCGACTA

In [4]:
# 1. Merge the predictions into your AnnData obs dataframe
adata_annotated = predictions.to_adata()

# 2. Check the counts of your cell types to see what you found
print(adata_annotated.obs["majority_voting"].value_counts())

# 3. Finally, save your annotated object safely to your Toshiba drive
output_path = "/media/sofia/TOSHIBA EXT/developing_human_brain/tumor_annotated.h5ad"
adata_annotated.write(output_path)
print(f"Successfully saved to {output_path}!")

... storing 'orig.ident' as categorical
... storing 'Sample_ID' as categorical
... storing 'Tumor_Type' as categorical
... storing 'condition' as categorical
... storing 'RNA_snn_res.0.8' as categorical
... storing 'seurat_clusters' as categorical
... storing 'DF.classification' as categorical
... storing 'RNA_snn_res.0.5' as categorical
... storing 'RNA_snn_res.1' as categorical
... storing 'RNA_snn_res.1.2' as categorical
... storing 'RNA_snn_res.1.5' as categorical
... storing 'RNA_snn_res.1.8' as categorical
... storing 'RNA_snn_res.2' as categorical
... storing 'celltype.1' as categorical
... storing 'by_group_celltypes' as categorical
... storing 'brain_location' as categorical
... storing 'var.features' as categorical


majority_voting
C4     23462
C3     15069
C35     3765
C30     1119
C13      718
C39      704
C0       347
C2       141
C20       98
Name: count, dtype: int64
Successfully saved to /media/sofia/TOSHIBA EXT/developing_human_brain/tumor_annotated.h5ad!


In [7]:
#model = celltypist.Model.load("/media/sofia/TOSHIBA EXT/developing_human_brain/BTS_atlas_CellTypist_model.pkl")

print(model.cell_types)
print(model.features)
print(model.description)

['C0' 'C1' 'C10' 'C11' 'C12' 'C13' 'C14' 'C15' 'C16' 'C17' 'C18' 'C19'
 'C2' 'C20' 'C21' 'C22' 'C23' 'C24' 'C25' 'C26' 'C27' 'C28' 'C29' 'C3'
 'C30' 'C31' 'C32' 'C33' 'C34' 'C35' 'C36' 'C37' 'C38' 'C39' 'C4' 'C40'
 'C5' 'C6' 'C7' 'C8' 'C9']
['A2M' 'AATBC' 'ABCA15P' ... 'RN7SL691P' 'RN7SL847P' 'RNA5SP200']
{'date': '2024-06-11 16:05:40.126323', 'details': '', 'url': '', 'source': '', 'version': '', 'number_celltypes': 41}


In [8]:
# 1. Ensure confidence scores are pulled into your AnnData object
adata_annotated = predictions.to_adata(insert_conf_by='majority_voting')

# 2. View summary statistics of the confidence scores
print(adata_annotated.obs['conf_score'].describe())

# 3. See average confidence score per cluster/cell type
print(adata_annotated.obs.groupby('majority_voting')['conf_score'].mean())

count    4.542300e+04
mean     4.475736e-01
std      4.060404e-01
min      2.375098e-13
25%      2.882382e-02
50%      3.393009e-01
75%      9.186005e-01
max      1.000000e+00
Name: conf_score, dtype: float64
majority_voting
C0     0.849041
C13    0.327146
C2     0.222007
C20    0.196395
C3     0.460989
C30    0.608970
C35    0.240890
C39    0.450763
C4     0.464483
Name: conf_score, dtype: float64
